# Import

In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
 
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import shap
 
print("=" * 60)
print("ML EXPERIMENT — XGBoost 30-Day Readmission Classifier")
print("=" * 60)


StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 5, Finished, Available, Finished, False)

ML EXPERIMENT — XGBoost 30-Day Readmission Classifier


##  Configure MLflow

In [3]:
EXPERIMENT_NAME = "HospitalReadmission_30Day"
mlflow.set_experiment(EXPERIMENT_NAME)

StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 6, Finished, Available, Finished, False)

<Experiment: artifact_location='sds://onelakecentralindia.pbidedicated.windows.net/d7ddf886-b4b2-42a7-bbe6-4f256e6573a2/a0887097-a342-4742-8474-98dd8c25c70a', creation_time=1777285399123, experiment_id='a0887097-a342-4742-8474-98dd8c25c70a', last_update_time=1777285399123, lifecycle_stage='active', name='HospitalReadmission_30Day', tags={}>

## Load Features

In [4]:
FEATURE_COLS = [
    "time_in_hospital","num_lab_procedures","num_procedures",
    "num_medications","number_diagnoses",
    "number_inpatient","number_emergency","number_outpatient",
    "prior_visits_total","is_high_prior_use",
    "has_prior_inpatient","has_prior_emergency",
    "is_long_stay","is_polypharmacy","is_complex_patient",
    "age_ord",
    "total_med_changes","total_meds_taken",
    "insulin_changed","insulin_increased",
    "A1C_tested","A1C_high","A1C_normal","glucose_tested",
    "is_diabetes_primary","has_circulatory_dx",
    "diag1_cat_idx","diag2_cat_idx","diag3_cat_idx",
    "gender_idx","race_idx",
]
TARGET = "readmitted_30d"
 
df_spark   = spark.read.format("delta").table("silver_features")
available  = [f for f in FEATURE_COLS if f in df_spark.columns]
print(f"Features available: {len(available)}")
 
df_pd = df_spark.select(["encounter_id","patient_nbr", TARGET] + available) \
               .toPandas()
print(f"Loaded {len(df_pd):,} patient records")
 
X = df_pd[available].fillna(0)
y = df_pd[TARGET]

StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 7, Finished, Available, Finished, False)

Features available: 31
Loaded 101,766 patient records


In [5]:
# Run this in a new cell after loading silver_features
df_pd = spark.read.format("delta").table("silver_features").toPandas()

check_cols = ['total_med_changes','insulin_changed','A1C_tested',
              'A1C_high','is_diabetes_primary','total_meds_taken']
for col in check_cols:
    vc = df_pd[col].value_counts()
    pct_nonzero = (df_pd[col] > 0).mean() * 100
    print(f"{col:<25} nonzero: {pct_nonzero:.1f}%  unique: {df_pd[col].nunique()}")

StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 8, Finished, Available, Finished, False)

total_med_changes         nonzero: 27.2%  unique: 5
insulin_changed           nonzero: 23.1%  unique: 2
A1C_tested                nonzero: 16.7%  unique: 2
A1C_high                  nonzero: 11.8%  unique: 2
is_diabetes_primary       nonzero: 8.7%  unique: 2
total_meds_taken          nonzero: 77.0%  unique: 7


## Class Imbalance Analysis

In [6]:
# The dataset has 4.5% positive rate (readmitted within 30 days).
# XGBoost's scale_pos_weight parameter compensates for this.
# Formula: (count negative) / (count positive)
pos_count  = y.sum()
neg_count  = len(y) - pos_count
scale_pw   = neg_count / pos_count
pos_rate   = y.mean()
 
print(f"\nClass distribution:")
print(f"  Positive (readmitted): {pos_count:,} ({pos_rate:.1%})")
print(f"  Negative (not):        {neg_count:,} ({1-pos_rate:.1%})")
print(f"  scale_pos_weight:      {scale_pw:.1f}")
 
# XGBoost Parameters ────────────────────────────
# XGB_PARAMS = dict(
#     n_estimators       = 500,
#     learning_rate      = 0.05,
#     max_depth          = 6,
#     subsample          = 0.8,
#     colsample_bytree   = 0.8,
#     min_child_weight   = 5,
#     gamma              = 0.1,
#     reg_alpha          = 0.1,
#     reg_lambda         = 1.0,
#     scale_pos_weight   = scale_pw,
#     eval_metric        = "auc",
#     use_label_encoder  = False,
#     random_state       = 42,
#     n_jobs             = -1,
#     verbosity          = 0,
# )

StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 9, Finished, Available, Finished, False)


Class distribution:
  Positive (readmitted): 11,357 (11.2%)
  Negative (not):        90,409 (88.8%)
  scale_pos_weight:      8.0


In [7]:
# early_stopping_rounds is NOT here — it requires eval_set
# which cross_val_score cannot provide

XGB_PARAMS = dict(
    n_estimators       = 2000,     # was 500 — give it room to run
    learning_rate      = 0.01,     # was 0.05 — smaller steps, better convergence
    max_depth          = 5,        # was 6 — slightly shallower to reduce overfit
    subsample          = 0.8,
    colsample_bytree   = 0.7,      # was 0.8 — less feature overlap per tree
    min_child_weight   = 10,       # was 5 — higher = less overfit on sparse positives
    gamma              = 0.2,      # was 0.1 — minimum split gain, prunes weak nodes
    reg_alpha          = 0.5,      # was 0.1 — more L1 sparsity
    reg_lambda         = 2.0,      # was 1.0 — more L2 smoothing
    scale_pos_weight   = scale_pw,
    # early_stopping_rounds = 80,    # was 50 — more patience before stopping
    eval_metric        = "auc",
    random_state       = 42,
    n_jobs             = -1,
    verbosity          = 0,
)

StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 10, Finished, Available, Finished, False)

## Cross-Validation

In [8]:
# Stratified k-fold ensures each fold has the same 11.2% positive rate.
# This gives a robust, unbiased AUC estimate before the final model fit.
print("Running 5-fold stratified cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = cross_val_score(
    XGBClassifier(**XGB_PARAMS), X, y,
    cv=cv, scoring="roc_auc", n_jobs=-1
)
print(f"CV AUC-ROC: {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}")

# cross_val_score calls .fit() with no eval_set — so no early stopping here.
# n_estimators=2000 is the fixed tree count for CV.
# Early stopping is applied only in the final model fit below.

StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 11, Finished, Available, Finished, False)

Running 5-fold stratified cross-validation...
CV AUC-ROC: 0.6431 ± 0.0057


##  Final Model + MLflow Run

In [9]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
 
with mlflow.start_run(run_name="XGBoost_Readmit30d_v2"):
 
    model = XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        early_stopping_rounds=80,   # ← only here, where eval_set exists
        verbose=False
    )
 
    # Predictions
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.35).astype(int)  # 35% threshold = HIGH tier
 
    # Metrics
    auc = roc_auc_score(y_te, y_prob)
    ap  = average_precision_score(y_te, y_prob)
 
    # SHAP explainability (sample 500 rows for speed)
    explainer   = shap.TreeExplainer(model)
    shap_vals   = explainer.shap_values(X_te.iloc[:500])
    fi_df = pd.DataFrame({
        "feature":    available,
        "mean_shap":  np.abs(shap_vals).mean(axis=0)
    }).sort_values("mean_shap", ascending=False)
 
    # Log to MLflow
    mlflow.log_param("n_features",        len(available))
    mlflow.log_param("train_size",        len(X_tr))
    mlflow.log_param("test_size",         len(X_te))
    mlflow.log_param("pos_rate",          round(pos_rate, 4))
    mlflow.log_param("scale_pos_weight",  round(scale_pw, 2))
    mlflow.log_params({f"xgb_{k}": v for k, v in XGB_PARAMS.items()})
    mlflow.log_metric("AUC_ROC",          round(auc, 4))
    mlflow.log_metric("Avg_Precision",    round(ap, 4))
    mlflow.log_metric("CV_AUC_mean",      round(cv_aucs.mean(), 4))
    mlflow.log_metric("CV_AUC_std",       round(cv_aucs.std(), 4))
    mlflow.log_metric("best_iteration",   model.best_iteration)
 
    # Save feature importance to MLflow artifact
    fi_path = "/tmp/feature_importance_readmit.csv"
    fi_df.to_csv(fi_path, index=False)
    mlflow.log_artifact(fi_path, artifact_path="feature_importance")
 
    # Register model
    mlflow.sklearn.log_model(
        model,
        artifact_path="xgb_readmission",
        registered_model_name="HospitalReadmission30d"
    )
 
    print(f"\n{'='*50}")
    print(f"AUC-ROC:       {auc:.4f}")
    print(f"Avg Precision: {ap:.4f}")
    print(f"Best iter:     {model.best_iteration}")
    print(f"\nTop 10 features by SHAP importance:")
    print(fi_df.head(10).to_string(index=False))

StatementMeta(, 73b5fd66-4ed3-4b46-a1e6-84a7b7ae94da, 12, Finished, Available, Finished, False)

`early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
[06:50:47] WARNING: /croot/xgboost-split_1713972711803/work/cpp_src/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
Setuptools is replacing distutils.
2026-04-28:06:51:12,672 ERROR    [shared_platform_utils.py:82] Create MLModel failed, status_code: 409, b'{"requestId":"5f785835-f012-4999-b4d9-28afe9d98a62","errorCode":"ItemDisplayNameAlreadyInUse","message":"Requested \'HospitalReadmission30d\' is already in use","isRetriable":false}'
Registered model 'HospitalReadmission30d' already exists. Creating a new version of this model...



AUC-ROC:       0.6445
Avg Precision: 0.1983
Best iter:     656

Top 10 features by SHAP importance:
            feature  mean_shap
   number_inpatient   0.201384
   time_in_hospital   0.088730
            age_ord   0.078177
 prior_visits_total   0.072166
has_prior_inpatient   0.064901
   total_meds_taken   0.061046
   number_diagnoses   0.058556
      diag1_cat_idx   0.056525
    num_medications   0.040684
 num_lab_procedures   0.032829


## Performance Interpretation

In [26]:
print(f"\n{'='*50}")
print("MODEL PERFORMANCE GUIDE")
print("="*50)
print(f"AUC-ROC: {auc:.4f}")
if auc >= 0.75:
    print("  → EXCELLENT for clinical readmission prediction")
elif auc >= 0.70:
    print("  → GOOD — competitive with published literature")
elif auc >= 0.60:
    print("  → ACCEPTABLE — better than no model, room to improve")
else:
    print("  → BELOW TARGET — review feature engineering")
 
print(f"\nProceed to → 05_risk_scoring.py")
 

StatementMeta(, d7ce0b3b-e4e3-428b-a565-c2a892457d0e, 40, Finished, Available, Finished, False)


MODEL PERFORMANCE GUIDE
AUC-ROC: 0.6445
  → ACCEPTABLE — better than no model, room to improve

Proceed to → 05_risk_scoring.py
